# Complete corrected perturbation-generation matrix

This notebook supports both a comprehensive one-step integration test and the genuine full attack-generation run. Both modes cover MVTec and VisA, eight loss/step/epsilon configurations under frozen WinCLIP prompts plus eight object-agnostic learnable-prompt counterparts, and the per-dataset, per-category, and per-image scopes.

The default `FULL_COMPLETE_RUN = False` runs every pipeline path with one PGD step and small sampling fractions. It checks dataset discovery, balanced deterministic splits, all attack scopes, artifacts, manifests, checksums, and packaging; its attack metrics are not meaningful. After it passes, set `FULL_COMPLETE_RUN = True` to use the real 500/800 steps and full configured sampling. Enable a Kaggle GPU and Internet. The notebook fetches the latest commit from the configured repository branch and records the resolved commit in the generated metadata.


In [ ]:
# User settings
REPOSITORY_URL = 'https://github.com/Parsagh05/adversarial-perturbation-generation.git'
REPOSITORY_REF = 'main'  # Fetch the latest commit from the main branch.

# False: complete one-step integration test. True: genuine full 500/800-step run.
FULL_COMPLETE_RUN = False

# The complete matrix. You may narrow these only when intentionally running subsets.
SOURCE_DATASETS = ('mvtec',)  # Only these datasets may optimize perturbations.
EVALUATION_DATASETS = ('mvtec', 'visa')  # Per-dataset deltas are delivered to these held-out datasets.
SCOPES = ('dataset', 'cross_dataset', 'category', 'image')
# RUN_SETUPS accepts 'all' or a comma-separated selection from these base setups:
#   steps500_eps2                 -> ce_focal_dice, 500 steps, epsilon 2/255
#   steps500_eps4                 -> ce_focal_dice, 500 steps, epsilon 4/255
#   steps800_eps2                 -> ce_focal_dice, 800 steps, epsilon 2/255
#   steps800_eps4                 -> ce_focal_dice, 800 steps, epsilon 4/255
#   steps500_eps2_margin_topk     -> margin_topk, 500 steps, epsilon 2/255
#   steps500_eps4_margin_topk     -> margin_topk, 500 steps, epsilon 4/255
#   steps800_eps2_margin_topk     -> margin_topk, 800 steps, epsilon 2/255
#   steps800_eps4_margin_topk     -> margin_topk, 800 steps, epsilon 4/255
# Append _gradnorm to any base setup for the gradient-normalized variant,
# which rescales each component gradient to unit norm before the 0.2/0.8
# weights and runs the combined loss mode only.
# PROMPT_SETUP controls which counterpart runs for each selected base setup:
#   'frozen'    -> frozen_prompt/<base_setup>/
#   'learnable' -> learnable_prompt/<base_setup>_learnable_prompt/
#   'both'      -> both counterparts (16 total when RUN_SETUPS='all')
# You may also provide an exact *_learnable_prompt setup ID.
RUN_SETUPS = 'all'
PROMPT_SETUP = 'both'  # Choose: 'frozen', 'learnable', or 'both'.

# Replace these sample paths after attaching the prompt-artifact Kaggle dataset.
LEARNABLE_PROMPT_MVTEC_CHECKPOINT = '/kaggle/input/datasets/parsaorbot/learned-prompts/prompts/mvtec/prompts_epoch15.pt'
LEARNABLE_PROMPT_VISA_CHECKPOINT = '/kaggle/input/datasets/parsaorbot/learned-prompts/prompts/visa/prompts_epoch15.pt'

SMOKE_TEST = not FULL_COMPLETE_RUN
SMOKE_STEPS = '1'
TEST_ATTACK_TRAIN_FRACTION = '0.05'
TEST_PER_IMAGE_EVALUATION_FRACTION = '0.02'
ATTACK_TRAIN_FRACTION = '1.0'
GPU = '0'

# T4-safe batch controls
PER_DATASET_BATCH_SIZE = '2'
PER_CATEGORY_EFFECTIVE_BATCH_SIZE = '8'
PER_CATEGORY_MICRO_BATCH_SIZE = '2'
PER_IMAGE_BATCH_SIZE = '2'
PER_IMAGE_EVALUATION_FRACTION = '1.0'

# Shared PGD behavior for every selected setup and scope
INITIAL_STEP_SIZE = '0.25/255'
STEP_SIZE_SCHEDULE = 'cosine'
STEP_SIZE_MIN_RATIO = '0.1'
FULL_TRAIN_CHECKPOINT_INTERVAL = '50'

# Normal-image local target: centred square covering 25% of each side
NORMAL_LOCAL_TARGET = 'fixed_region'
NORMAL_TARGET_REGION_FRACTION = '0.25'
NORMAL_TARGET_CENTER_X = '0.5'
NORMAL_TARGET_CENTER_Y = '0.5'


In [ ]:
# Runtime and dataset discovery
import os, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator and restart the session.')
print('GPU:', torch.cuda.get_device_name(0))

def shallow_input_directories():
    root = Path('/kaggle/input')
    for top in root.iterdir():
        if top.is_dir():
            yield top
            for child in top.iterdir():
                if child.is_dir():
                    yield child

def find_dataset_root(candidates, predicate, label):
    checked = set()
    for candidate in [*map(Path, candidates), *shallow_input_directories()]:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)
        if candidate.is_dir() and predicate(candidate):
            print(f'{label}: {candidate}')
            return candidate
    raise FileNotFoundError(f'{label} root was not found. Attach the dataset or add its mounted path to the candidates.')

valid_datasets = {'mvtec', 'visa'}
for name, selection in (('SOURCE_DATASETS', SOURCE_DATASETS), ('EVALUATION_DATASETS', EVALUATION_DATASETS)):
    if not selection or len(set(selection)) != len(selection) or not set(selection) <= valid_datasets:
        raise ValueError(f'{name} must contain unique values from: mvtec, visa')
PROTOCOL_DATASETS = tuple(dict.fromkeys((*SOURCE_DATASETS, *EVALUATION_DATASETS)))
valid_scopes = {'dataset', 'cross_dataset', 'category', 'image'}
if not SCOPES or len(set(SCOPES)) != len(SCOPES) or not set(SCOPES) <= valid_scopes:
    raise ValueError("SCOPES must contain unique values from: dataset, category, image")
MVTEC_ROOT = find_dataset_root([
    '/kaggle/input/mvtec-ad/mvtec_anomaly_detection',
    '/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection',
], lambda root: (root / 'bottle' / 'test').is_dir() and (root / 'bottle' / 'train' / 'good').is_dir(), 'MVTec') if 'mvtec' in PROTOCOL_DATASETS else Path('/kaggle/working/unused_mvtec')
VISA_ROOT = find_dataset_root([
    '/kaggle/input/visa-ad/VisA_20220922',
    '/kaggle/input/visa/VisA_20220922',
    '/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922',
], lambda root: (root / 'split_csv' / '1cls.csv').is_file() or (root / '1cls.csv').is_file(), 'VisA') if 'visa' in PROTOCOL_DATASETS else Path('/kaggle/working/unused_visa')


In [ ]:
# Fetch the latest commit from the requested branch/ref.
WORKING = Path('/kaggle/working')
REPO_ROOT = WORKING / 'adversarial-perturbation-generation'
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth', '1', 'origin', REPOSITORY_REF], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
RESOLVED_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Resolved repository commit:', RESOLVED_COMMIT)


In [ ]:
# Configure and run. Test and full outputs are deliberately separated.
source_tag = '-'.join(SOURCE_DATASETS)
evaluation_tag = '-'.join(EVALUATION_DATASETS)
dataset_tag = f'{source_tag}_to_{evaluation_tag}'
run_tag = 'test1step' if SMOKE_TEST else 'full'
OUTPUT_BASE = WORKING / f'attack_generation_{run_tag}_{dataset_tag}'
env = os.environ.copy()
env.update({
    'MVTEC_ROOT': str(MVTEC_ROOT),
    'VISA_ROOT': str(VISA_ROOT),
    'OUTPUT_BASE': str(OUTPUT_BASE),
    'GPU': GPU,
    'PYTHON_BIN': sys.executable,
    'USE_VENV': 'false',
    'RUN_SETUPS': RUN_SETUPS,
    'PROMPT_SETUP': PROMPT_SETUP,
    'LEARNABLE_PROMPT_MVTEC_CHECKPOINT': LEARNABLE_PROMPT_MVTEC_CHECKPOINT,
    'LEARNABLE_PROMPT_VISA_CHECKPOINT': LEARNABLE_PROMPT_VISA_CHECKPOINT,
    'SMOKE_TEST': str(SMOKE_TEST).lower(),
    'SMOKE_STEPS': SMOKE_STEPS,
    'INITIAL_STEP_SIZE': INITIAL_STEP_SIZE,
    'STEP_SIZE_SCHEDULE': STEP_SIZE_SCHEDULE,
    'STEP_SIZE_MIN_RATIO': STEP_SIZE_MIN_RATIO,
    'DIAGNOSTIC_INTERVAL': FULL_TRAIN_CHECKPOINT_INTERVAL,
    'NORMAL_LOCAL_TARGET': NORMAL_LOCAL_TARGET,
    'NORMAL_TARGET_REGION_FRACTION': NORMAL_TARGET_REGION_FRACTION,
    'NORMAL_TARGET_CENTER_X': NORMAL_TARGET_CENTER_X,
    'NORMAL_TARGET_CENTER_Y': NORMAL_TARGET_CENTER_Y,
    'PER_DATASET_BATCH_SIZE': PER_DATASET_BATCH_SIZE,
    'PER_CATEGORY_EFFECTIVE_BATCH_SIZE': PER_CATEGORY_EFFECTIVE_BATCH_SIZE,
    'PER_CATEGORY_MICRO_BATCH_SIZE': PER_CATEGORY_MICRO_BATCH_SIZE,
    'PER_IMAGE_BATCH_SIZE': PER_IMAGE_BATCH_SIZE,
    'PER_IMAGE_EVALUATION_FRACTION': PER_IMAGE_EVALUATION_FRACTION,
    'ATTACK_TRAIN_FRACTION': ATTACK_TRAIN_FRACTION,
    'SOURCE_DATASETS': ','.join(SOURCE_DATASETS),
    'EVALUATION_DATASETS': ','.join(EVALUATION_DATASETS),
    'RUN_PER_DATASET': str('dataset' in SCOPES).lower(),
    'RUN_CROSS_DATASET': str('cross_dataset' in SCOPES).lower(),
    'RUN_PER_CATEGORY': str('category' in SCOPES).lower(),
    'RUN_PER_IMAGE': str('image' in SCOPES).lower(),
})
if SMOKE_TEST:
    env.update({
        'ATTACK_TRAIN_FRACTION': TEST_ATTACK_TRAIN_FRACTION,
        'PER_IMAGE_EVALUATION_FRACTION': TEST_PER_IMAGE_EVALUATION_FRACTION,
        'DIAGNOSTIC_INTERVAL': '1',
    })

print('Mode:', 'COMPLETE ONE-STEP TEST' if SMOKE_TEST else 'GENUINE FULL RUN')
print('Source datasets:', SOURCE_DATASETS)
print('Evaluation datasets:', EVALUATION_DATASETS)
print('Scopes:', SCOPES)
print('Setups:', RUN_SETUPS)
print('Prompt setup:', PROMPT_SETUP)
print('Output:', OUTPUT_BASE)
GENERATOR_DIR = REPO_ROOT
subprocess.run(['bash', str(GENERATOR_DIR / 'train.sh')], cwd=GENERATOR_DIR, env=env, check=True)


In [ ]:
# Mandatory split, diagnostics, and packaging audit
import numpy as np
import pandas as pd

setup_roots = sorted(path for family in ('frozen_prompt', 'learnable_prompt') for path in (OUTPUT_BASE / 'setups' / family).glob('*') if path.is_dir())
if not setup_roots:
    raise RuntimeError('No setup output folders were produced.')

protocol_hashes = set()
for setup_root in setup_roots:
    train_path = setup_root / 'protocol' / 'attack_train_indices.csv'
    evaluation_path = setup_root / 'protocol' / 'evaluation_test_indices.csv'
    train = pd.read_csv(train_path)
    evaluation = pd.read_csv(evaluation_path)
    protocol = pd.concat([train, evaluation], ignore_index=True)
    label_counts = protocol.groupby(
        ['dataset', 'category', 'partition', 'label']
    ).size().unstack(fill_value=0)
    if set(label_counts.columns) != {0, 1} or not label_counts[0].eq(label_counts[1]).all():
        display(label_counts)
        raise RuntimeError(f'{setup_root.name}: protocol is not label-balanced.')
    if set(train.protocol_id) & set(evaluation.protocol_id):
        raise RuntimeError(f'{setup_root.name}: train/evaluation leakage detected.')
    display(protocol.groupby(['dataset', 'partition', 'label']).size().unstack(fill_value=0))

    manifests = sorted(setup_root.rglob('attack_manifest.csv'))
    if not manifests:
        raise RuntimeError(f'{setup_root.name}: no attack manifest was produced.')
    for manifest_path in manifests:
        manifest = pd.read_csv(manifest_path)
        protocol_hashes.update(manifest.protocol_split_sha256.astype(str).unique())
        expected_steps = int(SMOKE_STEPS) if SMOKE_TEST else (500 if 'steps500' in setup_root.name else 800)
        if set(manifest.optimization_steps.astype(int)) != {expected_steps}:
            raise RuntimeError(f'{manifest_path}: unexpected optimization step count.')
        expected_epsilon = (2 / 255) if 'eps2' in setup_root.name else (4 / 255)
        if not manifest.epsilon.astype(float).map(lambda value: abs(value - expected_epsilon) <= 1e-12).all():
            raise RuntimeError(f'{manifest_path}: unexpected epsilon.')
        expected_loss = 'margin_topk' if '_margin_topk' in setup_root.name else 'ce_focal_dice'
        if set(manifest.loss_formulation.astype(str)) != {expected_loss}:
            raise RuntimeError(f'{manifest_path}: unexpected loss formulation.')
        expected_prompt = 'learnable_object_agnostic' if setup_root.name.endswith('_learnable_prompt') else 'frozen_winclip'
        if set(manifest.prompt_mode.astype(str)) != {expected_prompt}:
            raise RuntimeError(f'{manifest_path}: unexpected prompt mode.')
        if expected_prompt == 'learnable_object_agnostic':
            if manifest.prompt_checkpoint_sha256.fillna('').str.len().ne(64).any():
                raise RuntimeError(f'{manifest_path}: invalid prompt checkpoint provenance.')

if len(protocol_hashes) != 1:
    raise RuntimeError('Selected setups did not reuse the same deterministic split.')

diagnostic_files = sorted(OUTPUT_BASE.rglob('optimization_diagnostics.csv'))
if not diagnostic_files:
    raise RuntimeError('No optimization diagnostics were produced.')
frames = []
for path in diagnostic_files:
    frame = pd.read_csv(path)
    frame.insert(0, 'diagnostics_file', str(path.relative_to(OUTPUT_BASE)))
    frames.append(frame)
diagnostics = pd.concat(frames, ignore_index=True)
metric_columns = ['initial_total_loss', 'final_total_loss', 'total_loss_reduction']
if not np.isfinite(diagnostics[metric_columns].apply(pd.to_numeric, errors='coerce')).all().all():
    raise RuntimeError('Optimization diagnostics contain non-finite loss values.')
display(diagnostics.groupby(['scope', 'loss_formulation', 'loss_mode'])[metric_columns].mean())

passed = diagnostics['convergence_check_passed'].astype(str).str.lower().eq('true')
if not passed.all():
    display(diagnostics[~passed])
    if SMOKE_TEST:
        print('TEST NOTE: convergence is not required after only one PGD step; artifact checks continue.')
    else:
        raise RuntimeError('Some full-run conditions failed the convergence check.')

archives = sorted(OUTPUT_BASE.rglob('canonical_clip_*.zip'))
if not archives:
    raise RuntimeError('No ZIP archive was produced.')
for archive in archives:
    print(f'{archive.relative_to(OUTPUT_BASE)}: {archive.stat().st_size / 2**30:.3f} GiB')
full_archive = OUTPUT_BASE / 'full_outputs.zip'
if not full_archive.is_file():
    raise RuntimeError('The combined full_outputs.zip archive was not produced.')
print(f'{full_archive.relative_to(OUTPUT_BASE)}: {full_archive.stat().st_size / 2**30:.3f} GiB')
print('AUDIT PASSED.' if not SMOKE_TEST else 'COMPLETE ONE-STEP PIPELINE TEST PASSED. Do not publish test archives.')
